In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

In [4]:
# Copy the folder to your drive and explore, so you can load data and weight of your own way
HOME_PATH = '/content/drive/My Drive/LLM-Lab/Lab 1 - Language Models with RNN/Lab 1.2'
DATA_PATH = os.path.join(HOME_PATH, 'data')

WEIGHT_PATH = os.path.join(HOME_PATH, 'weight')
if not os.path.isdir(WEIGHT_PATH):
    os.makedirs(WEIGHT_PATH)

## Preprocess data

## Load data

In [5]:
with open(os.path.join(DATA_PATH, 'last_name.txt')) as f:
    data = f.read().split('\n')[:-1]

## Tokenize

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
tokenizer = Tokenizer(char_level=True, filters='', split='', oov_token='*')

In [9]:
pad_token = 0
unknow_token = 1

In [10]:
tokenizer.fit_on_texts(data)

In [11]:
seq_all = tokenizer.texts_to_sequences(data)

In [12]:
len_all = [len(x) for x in seq_all]
max_len = max(len_all)

In [13]:
seq_all = pad_sequences(seq_all, maxlen=max_len)

In [14]:
X_all = seq_all[:,:-1]
y_all = seq_all[:,1:]

In [15]:
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

In [16]:
import torch
torch.save(tokenizer, os.path.join(DATA_PATH, 'last_name_tokenizer.h5'))

## Data loader

In [17]:
import torch
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset

In [18]:
def build_dataset_from_tensors(X, y):
    ds = TensorDataset(X, y)
    return ds

In [19]:
X_train = torch.tensor(X_train).long()
y_train = torch.tensor(y_train).long()
X_val = torch.tensor(X_val).long()
y_val = torch.tensor(y_val).long()

In [20]:
train_ds = build_dataset_from_tensors(X_train, y_train)
val_ds = build_dataset_from_tensors(X_val, y_val)

In [21]:
batch_size = 256
shuffle = True

In [22]:
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=shuffle)

## Model

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [24]:
from tqdm import tqdm

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### GRADED FUNCTION: Model

class Model(nn.Module):
    def __init__(self, vocab_size, embeding_size, hidden_size):
        super(Model, self).__init__()
        self.embeding = nn.Embedding(vocab_size, embeding_size)

        ### START CODE HERE ###
        # Initialize LSTM layer with 2 layers, batch_first=True, dropout=0.1
        self.lstm = nn.LSTM(input_size=embeding_size,
                            hidden_size=hidden_size,
                            num_layers=2,
                            batch_first=True,
                            dropout=0.1)

        # Initialize linear layer to transform from hidden_size to vocab_size
        self.linear = nn.Linear(hidden_size, vocab_size)
        ### END CODE HERE ###

    def forward(self, x, hidden_state=None):
        '''
        Performs forward pass through the model

        Arguments:
        x -- input tensor with shape (batch_size, sequence_length)
        hidden_state -- hidden state from previous step (optional)

        Returns:
        x -- output tensor with shape (batch_size, sequence_length, vocab_size)
        hidden_state -- new hidden state
        '''
        # x: BxS (Batch x Sequence)

        ### START CODE HERE ###
        # Step 1: Pass input through embedding layer
        x = self.embeding(x) # BxSxE (Batch x Sequence x Embedding)

        # Step 2: Pass through LSTM layer
        # Check if hidden_state is None, then don't pass it to LSTM
        if hidden_state is None:
            x, hidden_state = self.lstm(x)
        else:
            x, hidden_state = self.lstm(x, hidden_state)

        # Step 3: Apply ReLU activation function
        x = F.relu(x)

        # Step 4: Pass through linear layer for final output
        x = self.linear(x) # BxSxV (Batch x Sequence x Vocab_size)
        ### END CODE HERE ###

        return x, hidden_state

    def predict(self, x, hidden_state=None):
        '''
        Performs prediction with softmax

        Arguments:
        x -- input tensor
        hidden_state -- hidden state from previous step (optional)

        Returns:
        x -- probability distribution over vocab
        hidden_state -- new hidden state
        '''
        ### START CODE HERE ###
        # Step 1: Call forward function
        x, hidden_state = self.forward(x, hidden_state)

        # Step 2: Apply softmax to get probability distribution
        x = F.softmax(x, dim=-1) # BxSxV
        ### END CODE HERE ###

        return x, hidden_state

In [26]:
def forward_and_loss(model, x, y, loss_fn, pad_token):
    '''
    Performs forward pass and computes loss

    Arguments:
    model -- the neural network model
    x -- input tensor with shape (batch_size, sequence_length)
    y -- target tensor with shape (batch_size, sequence_length)
    loss_fn -- loss function (e.g., CrossEntropyLoss)
    pad_token -- padding token ID to ignore in loss calculation

    Returns:
    out -- model output with shape (batch_size, sequence_length, vocab_size)
    loss -- computed loss value
    '''

    ### START CODE HERE ###
    # Step 1: Forward pass through the model
    out, hidden_state = model(x)

    # Step 2: Compute loss using loss function
    # Reshape output from (B, S, V) to (B*S, V) for loss calculation
    # Reshape target from (B, S) to (B*S) for loss calculation
    output_flat = out.contiguous().view(-1, out.size(-1))
    target_flat = y.contiguous().view(-1)
    # Use ignore_index parameter to ignore pad_token in loss
    loss = loss_fn(out.contiguous().view(-1, out.shape[-1]), y.contiguous().view(-1), ignore_index=pad_token)
    ### END CODE HERE ###

    return out, loss

In [27]:
def train_model(model, optim, train_iter, loss_fn, pad_token, weight_path=None, device=None):
    '''
    Train the model for one epoch

    Arguments:
    model -- the neural network model to train
    optim -- optimizer for updating model parameters
    train_iter -- training data iterator
    loss_fn -- loss function to compute training loss
    pad_token -- padding token ID to ignore in loss calculation
    weight_path -- path to save model weights (optional)
    device -- device to run training on (optional)

    Returns:
    total_loss -- accumulated loss over the epoch
    '''
    total_loss = 0.0

    ### START CODE HERE ###
    # Step 1: Set model to training mode
    model.train()
    ### END CODE HERE ###

    with tqdm(total=len(train_iter)) as pbar:
        for x, y in train_iter:
            ### START CODE HERE ###
            # Step 2: Move data to device if CUDA is available
            # Check if device is not None and device type is 'cuda'
            if device is not None and device.type == 'cuda':
                x = x.to(device)
                y = y.to(device)
            ### END CODE HERE ###

            ### START CODE HERE ###
            # Step 3: Clear gradients from previous step
            optim.zero_grad()
            ### END CODE HERE ###

            _, loss = forward_and_loss(model, x, y, loss_fn, pad_token=pad_token)

            ### START CODE HERE ###
            # Step 4: Compute gradients via backpropagation
            loss.backward()

            # Step 5: Update model parameters
            optim.step()
            ### END CODE HERE ###

            total_loss += loss.item()

            pbar.update(1)
            pbar.set_description("%-10s = %.6f  " % ('loss', total_loss))

    ### START CODE HERE ###
    # Step 6: Save model weights if weight_path is provided
    # Check if weight_path is not None
    if weight_path is not None:
        state = {
            "model": model.state_dict(),
            "optim": optim.state_dict()
        }
        torch.save(state, weight_path)
    ### END CODE HERE ###

    return total_loss

In [28]:
def evaluate_model(model, val_iter, pad_token, device=None):
    model.eval()
    with torch.no_grad(), tqdm(total=len(val_iter)) as pbar:
        total_loss = 0.0

        for x, y in val_iter:
            if device is not None and device.type=='cuda':
                x = x.cuda()
                y = y.cuda()

            _, loss = forward_and_loss(model, x, y, F.cross_entropy, pad_token=pad_token)

            total_loss += loss.item()

            pbar.update(1)
            pbar.set_description("%-10s = %.6f  " % ('val_loss', total_loss))

    return total_loss

## Training

In [29]:
vocab_size = len(tokenizer.word_index) + 1
embedding_size = 200
hidden_size = 256
learning_rate = 0.0001
loss_fn = F.cross_entropy

In [30]:
device = torch.torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [31]:
model = Model(vocab_size, embedding_size, hidden_size)

In [32]:
if device.type=='cuda':
    model = model.cuda()

In [33]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

In [34]:
WEIGHT_PATH = os.path.join(HOME_PATH, 'ln_weight')
if not os.path.isdir(WEIGHT_PATH):
    os.makedirs(WEIGHT_PATH)

In [ ]:
# logging.info('\n' + '*'*50 + '\n' + "Start logging\nvocab_size=%d\nembedding_size=%d\nhidden_size=%d\nlearning_rate=%d\nbatch_size=%d\n" \
#              % (vocab_size, embedding_size, hidden_size, learning_rate, batch_size))

In [35]:
num_epoch = 300

In [ ]:
for i in range(1, num_epoch+1):
    weight_path = None
    if i%10==0:
        weight_path = os.path.join(WEIGHT_PATH, 'epoch_%02d.h5' % i)
    print("\nEpoch %02d" % i, flush=True)
    train_loss = train_model(model, optimizer, train_dl, loss_fn, pad_token, weight_path, device)
    val_loss = evaluate_model(model, val_dl, pad_token, device)


Epoch 01


val_loss   = 1895.789299  : 100%|██████████| 843/843 [00:07<00:00, 119.48it/s]


Epoch 02



val_loss   = 1801.282795  : 100%|██████████| 843/843 [00:07<00:00, 116.81it/s]


Epoch 03



val_loss   = 1758.223084  : 100%|██████████| 843/843 [00:06<00:00, 124.61it/s]

... (438 outputs left out)


Epoch 223



val_loss   = 1579.907142  : 100%|██████████| 843/843 [00:07<00:00, 112.36it/s]


Epoch 224



loss       = 2763.804093  :  46%|████▌     | 1534/3370 [00:32<00:38, 47.39it/s]